# Robust location under contamination *(MCP stress test)*
**Question:** estimate centre of a distribution when a fraction $\varepsilon$ of points
come from a heavy-tail contaminant—not a packaged tutorial; custom sweep.
Compare: plain mean vs trimmed mean (10% / 20%) vs median; plot error vs $\varepsilon$.

In [1]:
# setup — numpy only, deterministic seed
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

RNG = np.random.default_rng(1729)
n = 140
truth = 0.0


In [2]:
# mixture sweep — minimal numpy, RMS error vs truth mu=truth
EPS = np.linspace(0.0, 0.48, 13)
REPS = 30
methods = ('mean', 'median', 'trim10', 'trim20')

def contaminated_sample(nn, eps, rng):
    base = rng.standard_normal(nn)
    k = int(round(eps * nn))
    if k < 1:
        return base
    ix = rng.choice(nn, size=k, replace=False)
    contaminated = rng.standard_t(df=3, size=k) * 15.0
    out = base.copy()
    out[ix] = contaminated
    return out


def est_map(arr):
    a = np.sort(arr)
    m = int(arr.size)
    mn = arr.mean()
    med = float(np.median(arr))

    def trim(lo_frac, hi_frac):
        lo = int(np.floor(lo_frac * m))
        hi = int(np.ceil((1 - hi_frac) * m))
        if hi <= lo + 1:
            slic = arr
        else:
            slic = a[lo:hi]
        return float(np.mean(slic)) if slic.size else float('nan')

    return {'mean': mn, 'median': med, 'trim10': trim(0.1, 0.1), 'trim20': trim(0.2, 0.2)}


rms_vs_eps = {k: [] for k in methods}
for eps in EPS:
    sq = {k: [] for k in methods}
    for _ in range(REPS):
        x = contaminated_sample(n, eps, RNG)
        e = est_map(x)
        for k in methods:
            sq[k].append((e[k] - truth) ** 2)
    for k in methods:
        rms_vs_eps[k].append(float(np.sqrt(np.mean(np.array(sq[k])))))
diag_eps = EPS[len(EPS) // 2]
dx = contaminated_sample(n, diag_eps, RNG)
print("diag_eps", float(diag_eps), "skew", float(((dx - dx.mean()) ** 3).mean()), "std", float(dx.std()), "median", float(np.median(dx)))
rms_vs_eps_numpy = np.column_stack(tuple(rms_vs_eps[k] for k in methods))


diag_eps 0.24 skew 2615.8820970709735 std 9.985893052643489 median 0.04423192597420633


In [3]:
from pathlib import Path
ROOT = Path(r"benchmarks/case_studies")
PX = ROOT / "longtest_rmse_curve.png"
fig, ax = plt.subplots(figsize=(5.5, 3.8))
for j, lbl in enumerate(methods):
    ax.semilogy(EPS, rms_vs_eps_numpy[:, j], marker="o", ms=3, lw=1.6, label=lbl)
ax.set_xlabel("contamination ε")
ax.set_ylabel("RMSE vs truth")
ax.grid(True, ls=":")
ax.legend()
title = Path(__file__).name if '__file__' in dir() else 'longtest'
ax.set_title("Location estimators breakdown curve")
fig.tight_layout()
fig.savefig(PX, dpi=120)
plt.close(fig)
print("saved", PX.resolve())


saved C:\Users\goela\Projects\LLM_Jupyter\demos\longtest_rmse_curve.png
